# Classroom Pipeline Lab · ทดลองรันทีละเดือน

Notebook นี้ใช้ Source ชุดเดียวกับหน้า Test Pipeline: Run 1 = มกราคม 2569 และปุ่ม “เติมเดือนถัดไป” จะเพิ่มครั้งละ 1 เดือนจนถึงตุลาคม

กด **Runtime → Run all** เพื่อเตรียมเครื่องมือ จากนั้นใช้ปุ่มในส่วนควบคุมด้านล่าง ผลแต่ละ Run จะแสดงสถานะ 5 ขั้น, จำนวนก่อน–หลัง, ผลตรวจ และตัวอย่างแถวจาก Warehouse

Colab ใช้ SQLite / Parquet / DuckDB ใน Runtime ของผู้รันเอง ไม่เชื่อม Neon และไม่เปลี่ยนข้อมูลบนเว็บ

## 1. เตรียมโปรเจกต์และ Source จาก GitHub

In [ ]:
#@title 1 — เตรียมโปรเจกต์และ Source (ยังไม่เริ่มรัน Pipeline)
from pathlib import Path
import os, sys, subprocess, importlib, json, time, sqlite3, shutil
from IPython.display import display, Markdown, clear_output
import pandas as pd

REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo"
REPO_REF = "feature/classroom-demo-evidence-foundation"
PROJECT = "TikTok_Shop_Pipeline_Demo"
ROOT = Path("/content") / PROJECT if Path("/content").is_dir() else Path.cwd() / PROJECT
if not (ROOT / "pipeline" / "orchestrator.py").is_file():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError("โฟลเดอร์มีไฟล์ที่ไม่ใช่โปรเจกต์ตัวอย่าง กรุณาใช้ Runtime ใหม่")
    subprocess.check_call(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF, REPO_URL, str(ROOT)])
required=["requirements.txt","pipeline/orchestrator.py","pipeline/generate_classroom_source.py"]
missing=[x for x in required if not (ROOT/x).is_file()]
if missing: raise RuntimeError("โครงสร้างโปรเจกต์ไม่ครบ: "+", ".join(missing))
try:
    import duckdb, pyarrow, pandas
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-r",str(ROOT/"requirements-ml.txt")])
os.environ.pop("NEON_DEMO_DATABASE_URL",None)
os.chdir(ROOT)
root_path=str(ROOT.resolve())
sys.path=[p for p in sys.path if p!=root_path]; sys.path.insert(0,root_path)
for name in list(sys.modules):
    if name=="pipeline" or name.startswith("pipeline."): del sys.modules[name]
importlib.invalidate_caches()
from pipeline.generate_classroom_source import build, BATCHES, BATCH_MONTHS
from pipeline.orchestrator import run_batch, reset
from pipeline.demo_tests import signature, error_suite
from pipeline.quality import verify
import duckdb

BASE=ROOT/"output"/"colab_classroom"
SOURCE=BASE/"source"
WORKSPACE=BASE/"main"
ERROR_WORKSPACE=BASE/"error_lab"
SESSION_FILE=BASE/"session.json"
BASE.mkdir(parents=True,exist_ok=True)
build(SOURCE)
print("พร้อมใช้ Code สาขา:", REPO_REF)
print("Source:",len(BATCHES),"เดือน · มกราคม–ตุลาคม 2569 · 21 ตารางต่อเดือน")
print("ผลการรันจะอยู่ใน Colab Runtime เท่านั้น · ไม่เชื่อม Neon")
print("การเตรียมเสร็จแล้ว แต่ยังไม่มีการรัน Pipeline จนกว่าจะกดปุ่มเริ่ม")

## 2. กดปุ่มเพื่อเริ่มหรือเติมข้อมูล

In [ ]:
#@title 2 — ควบคุมการทดลอง
import ipywidgets as widgets

output=widgets.Output()
error_output=widgets.Output()
start_button=widgets.Button(description="เริ่มรอบใหม่และรัน Run 1 · มกราคม",button_style="primary",icon="play")
add_button=widgets.Button(description="เติมข้อมูลเดือนถัดไป",button_style="success",icon="plus")
clear_button=widgets.Button(description="เคลียร์ผลทดลองใน Colab",button_style="warning",icon="trash")
error_button=widgets.Button(description="ทดสอบ Error แยก",button_style="",icon="exclamation-triangle")
error_case=widgets.Dropdown(options=[("รหัสสินค้าไม่อยู่ใน Master","E01"),("ไฟล์ Source หายไป","E02"),("เงื่อนไข Voucher ไม่ผ่าน","T01"),("ช่วง Campaign ซ้อนกัน","T02"),("Warehouse ยกเลิกการบันทึก","L01"),("ส่งข้อมูลชุดเดิมซ้ำ","L02")],description="กรณี:")
controls=widgets.HBox([start_button,add_button,clear_button])
display(Markdown("**วิธีใช้:** เริ่มรอบใหม่จะเคลียร์เฉพาะผลเก่าใน Colab แล้วรันเดือนมกราคมทันที · ปุ่มเติมจะรันเดือนถัดไปเพียงเดือนเดียว · หากไม่ผ่าน กดปุ่มเดิมเพื่อลองเดือนเดิมอีกครั้ง · Error Lab แยกพื้นที่จากข้อมูลปกติ"))
display(controls)
display(widgets.HBox([error_case,error_button]))
display(output)
display(error_output)

def _snapshot():
    orders=0; items=0; sales=0.0
    if (WORKSPACE/"er1.sqlite").is_file():
        with sqlite3.connect(WORKSPACE/"er1.sqlite") as db: orders=int(db.execute('SELECT COUNT(*) FROM Fact_Orders').fetchone()[0])
    if (WORKSPACE/"er2.duckdb").is_file():
        with duckdb.connect(str(WORKSPACE/"er2.duckdb"),read_only=True) as db:
            items,sales=db.execute('SELECT COUNT(*),COALESCE(SUM(Net_Revenue),0) FROM Fact_Sales_Order_Item').fetchone()
    return {"orders":orders,"items":int(items),"sales":float(sales)}

def _show_batch(batch,before):
    month=BATCH_MONTHS[BATCHES.index(batch)].strftime("%Y-%m")
    print("เดือน Source:",month,"· Batch:",batch)
    try:
        result=run_batch(WORKSPACE,batch,source_root=SOURCE)
        after=_snapshot()
        rows=[]
        labels={"ingest":"Source → Database","extract":"Database → Data Lake","transform":"จัดรูปข้อมูลใน Data Lake","load":"Data Lake → Warehouse","validate":"ตรวจผลทั้งเส้นทาง"}
        for stage,info in result.get("metrics",{}).items():
            detail=info.get("result",{})
            records=next((detail.get(k) for k in ("input_records","raw_records","total_loaded","changed_sales") if detail.get(k) is not None),None)
            rows.append({"ขั้นตอน":labels.get(stage,stage),"ผล":"ผ่านแล้ว","เวลา (วินาที)":round(float(info.get("elapsed_seconds",0)),2),"จำนวนรายการที่ประมวลผล":records if records is not None else "ดูหลักฐานขั้นตอน"})
        display(pd.DataFrame(rows))
        print("ออเดอร์ในฐานข้อมูล: ก่อน",before["orders"],"→ หลัง",after["orders"],"· เพิ่ม",after["orders"]-before["orders"])
        print("รายการขายใน Warehouse: ก่อน",before["items"],"→ หลัง",after["items"],"· เพิ่ม",after["items"]-before["items"])
        print("ยอดขายสุทธิใน Warehouse: ก่อน ฿{:,.2f} → หลัง ฿{:,.2f}".format(before["sales"],after["sales"]))
        print("ผล Validation: ผ่าน",result.get("functional_passes","—"),"· ไม่ผ่าน",result.get("functional_fails","—"))
        with duckdb.connect(str(WORKSPACE/"er2.duckdb"),read_only=True) as db:
            sample=db.execute('SELECT Source_Order_Item_ID,Net_Revenue,COGS FROM Fact_Sales_Order_Item ORDER BY Source_Order_Item_ID LIMIT 5').df()
        print("ตัวอย่างข้อมูลจริงที่อ่านจาก Warehouse:"); display(sample)
        session={"next_index":BATCHES.index(batch)+1,"last_batch":batch,"last_month":month,"last_status":"SUCCESS"}
        SESSION_FILE.write_text(json.dumps(session,ensure_ascii=False,indent=2),encoding="utf-8")
        print("เดือนนี้เสร็จแล้ว · กด “เติมข้อมูลเดือนถัดไป” เพื่อเพิ่มอีก 1 เดือน")
    except Exception as exc:
        print("Run ไม่ผ่านในขั้นตอนใดขั้นตอนหนึ่ง")
        print("รายละเอียด:",type(exc).__name__,str(exc))
        print("ข้อมูลชุดนี้ยังไม่ถูกทำเครื่องหมายว่าเสร็จ · แก้ Source หากจำเป็น แล้วกดปุ่มเดิมเพื่อลองเดือนนี้อีกครั้ง")

def _start(_):
    start_button.disabled=True; add_button.disabled=True
    with output:
        clear_output(wait=True)
        reset(WORKSPACE)
        SESSION_FILE.write_text(json.dumps({"next_index":0,"last_status":"EMPTY"},ensure_ascii=False),encoding="utf-8")
        print("เริ่ม Session ใหม่ · เคลียร์ผลเฉพาะใน Colab แล้ว")
        _show_batch(BATCHES[0],{"orders":0,"items":0,"sales":0.0})
    start_button.disabled=False; add_button.disabled=False

def _add(_):
    start_button.disabled=True; add_button.disabled=True
    with output:
        clear_output(wait=True)
        if not SESSION_FILE.is_file():
            print("ยังไม่มี Session · กด “เริ่มรอบใหม่และรัน Run 1” ก่อน")
        else:
            session=json.loads(SESSION_FILE.read_text(encoding="utf-8"))
            index=int(session.get("next_index",0))
            if index>=len(BATCHES): print("Source ครบทุกเดือนถึงตุลาคม 2569 แล้ว · หากต้องการเริ่มใหม่ กดปุ่มเริ่มรอบใหม่")
            else: _show_batch(BATCHES[index],_snapshot())
    start_button.disabled=False; add_button.disabled=False

def _clear(_):
    reset(WORKSPACE)
    if ERROR_WORKSPACE.exists(): reset(ERROR_WORKSPACE)
    if SESSION_FILE.exists(): SESSION_FILE.unlink()
    with error_output: clear_output(wait=True)
    with output:
        clear_output(wait=True)
        print("เคลียร์ผล Pipeline และ Error Lab ใน Colab แล้ว · Source ต้นทางยังอยู่ · Neon และเว็บไม่ได้เปลี่ยน")

def _error(_):
    error_button.disabled=True
    with error_output:
        clear_output(wait=True)
        print("ทดสอบกรณี:",error_case.label,"· ใช้พื้นที่ Error Lab แยกจาก Warehouse หลัก")
        try:
            result=error_suite(ERROR_WORKSPACE,cases=[error_case.value])
            display(result[["case","stage","detected","retry_from","before","after","passed"]])
            row=result.iloc[0]
            print("ระบบตรวจพบ:",row["detected"],"· ขั้นที่เกี่ยวข้อง:",row["stage"],"· ผลหลังการกู้คืน:","ผ่าน" if bool(row["passed"]) else "ไม่ผ่าน")
            print("ผล Error Test นี้ไม่เปลี่ยนข้อมูลจาก Run 1/เดือนที่เติม · กด “เคลียร์ผลทดลองใน Colab” เมื่อต้องการลบผลทั้งหมด")
        except Exception as exc: print("Error Test ทำงานไม่สำเร็จ:",type(exc).__name__,str(exc))
    error_button.disabled=False

start_button.on_click(_start)
add_button.on_click(_add)
clear_button.on_click(_clear)
error_button.on_click(_error)
if SESSION_FILE.is_file():
    _session=json.loads(SESSION_FILE.read_text(encoding="utf-8"))
    print("Session นี้มีอยู่แล้ว · เดือนถัดไป:",BATCH_MONTHS[min(int(_session.get("next_index",0)),len(BATCH_MONTHS)-1)].strftime("%Y-%m") if int(_session.get("next_index",0))<len(BATCH_MONTHS) else "Source ครบแล้ว")
else:
    print("พร้อมเริ่ม · ยังไม่มีการรันข้อมูล")

## 3. Feature และ ML (ทางเลือก)

ML ใช้ Warehouse หลังเติมข้อมูลแล้ว ไม่ได้ใช้ยอดที่กำหนดไว้ล่วงหน้า Feature คือประวัติยอดรายสัปดาห์และแผนโฆษณา/ส่วนลดที่ทราบก่อนสัปดาห์เป้าหมาย

การฝึกจะแสดงจำนวนสัปดาห์จริงและเริ่มได้เมื่อมีสัปดาห์สมบูรณ์อย่างน้อย 16 สัปดาห์: Train ขั้นต่ำ 6 + Validation 2 รอบ รอบละ 3 + Test ที่กันไว้ 4 สัปดาห์ ข้อมูลรายเดือนหนึ่งเดือนจึงไม่เท่ากับหนึ่งตัวอย่าง ML โดยตรง

## 4. ทดสอบ Error (ตัวเลือก)

Error Test เป็นพื้นที่แยก เพื่อให้เห็นสาเหตุ จุดตรวจพบ และผลหลังระบบกู้คืน โดยไม่ทำให้การรันข้อมูลปกติล้มเหลว

## 5. ข้อจำกัดการสาธิต

- Source รายเดือนทั้ง 10 ชุดมีไฟล์ CSV 21 ตารางและหัวคอลัมน์ชุดเดียวกันตาม ER1
- Run 1 เริ่มจากมกราคม 2569; Run ถัดไปเพิ่มเดือนละหนึ่งชุดในฐานข้อมูลเดิม
- “เคลียร์” ใน Notebook ลบผลเฉพาะ Runtime ของผู้เปิด Notebook; ไม่ลบ Source บน GitHub, Neon, หรือผลของเพื่อนคนอื่น
- Run บนเว็บใช้ GitHub Actions และ Neon Lab ร่วมกัน ส่วน Colab เป็นสำเนาแยกใน Runtime เพื่อเป็นช่องทางสำรอง